# 03 - Baseline Regresi Logistik

Tujuan: membuat baseline dengan **fitur asli** dan mengevaluasinya hanya pada
validation. Test April-Juni 2026 hanya dibuat sebagai split dan dihitung jumlah
baris serta rasio targetnya; tidak dipakai untuk prediksi atau evaluasi.

Aturan:
- Split berdasarkan `tanggal_pickup`.
- Imputer, scaler, dan one-hot encoder di-fit hanya pada train.
- Tidak memakai fitur turunan dan kolom `flag_*`; itu masuk Tahap B.

In [1]:
from pathlib import Path
import sys

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

data = pd.read_csv(ROOT / "data" / "clean.csv", parse_dates=["tanggal_pickup"])

train = data.loc[data["tanggal_pickup"] <= "2025-12-31"].copy()
valid = data.loc[data["tanggal_pickup"].between("2026-01-01", "2026-03-31")].copy()
test = data.loc[data["tanggal_pickup"].between("2026-04-01", "2026-06-30")].copy()

# Test hanya dipakai untuk hitungan di tabel ini, bukan untuk model/evaluasi.
ringkasan = pd.DataFrame(
    {
        "bagian": ["train", "validation", "test"],
        "baris": [len(train), len(valid), len(test)],
        "rasio_terlambat": [
            train["terlambat"].mean(),
            valid["terlambat"].mean(),
            test["terlambat"].mean(),
        ],
    }
)
ringkasan

,bagian,baris,rasio_terlambat
0,train,26633,0.177787
1,validation,6623,0.218783
2,test,6744,0.139235


## Fitur asli

Fitur yang dipakai hanya kolom asli saat pickup:
- numerik: `jam_pickup`, `jarak_km`, `jumlah_transit`, `janji_hari`, `berat_kg`,
  `volume_hub_hari`
- kategorikal: `kota_tujuan`, `layanan`, `metode_bayar`, `cuaca_saat_pickup`

`id_resi` tidak dipakai karena hanya pengenal. `tanggal_pickup` dipakai untuk
split, bukan fitur. Kolom `flag_*` tidak dipakai karena sudah fitur turunan.

In [2]:
NUM_COLS = [
    "jam_pickup",
    "jarak_km",
    "jumlah_transit",
    "janji_hari",
    "berat_kg",
    "volume_hub_hari",
]
CAT_COLS = [
    "kota_tujuan",
    "layanan",
    "metode_bayar",
    "cuaca_saat_pickup",
]

X_train = train[NUM_COLS + CAT_COLS]
y_train = train["terlambat"]
X_valid = valid[NUM_COLS + CAT_COLS]
y_valid = valid["terlambat"]

print("Jumlah fitur asli:", len(NUM_COLS) + len(CAT_COLS))
print("Baris train:", len(X_train), "| baris validation:", len(X_valid))

Jumlah fitur asli: 10
Baris train: 26633 | baris validation: 6623


## Pipeline

Semua tahap preprocessing berada di dalam `Pipeline`, sehingga saat `fit`
hanya melihat train. Nilai hilang numerik diisi median train; kategorikal diisi
modus train; kategori baru saat validation diabaikan oleh encoder.

In [3]:
numeric_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("numeric", numeric_pipeline, NUM_COLS),
        ("categorical", categorical_pipeline, CAT_COLS),
    ]
)

model = Pipeline(
    [
        ("preprocessor", preprocessor),
        ("logistic", LogisticRegression(max_iter=2000, random_state=42)),
    ]
)

model.fit(X_train, y_train)
print("Model selesai dilatih pada train.")
print("Jumlah fitur setelah one-hot:", model.named_steps["preprocessor"].get_feature_names_out().shape[0])

Model selesai dilatih pada train.
Jumlah fitur setelah one-hot: 30


## Evaluasi validation

PR-AUC dihitung dengan `average_precision_score`. Model tanpa kemampuan
prediksi memiliki PR-AUC sekitar sebesar proporsi positif di validation, jadi
angka prevalensi validation dipakai sebagai pembanding *no-skill*.

In [4]:
proba_valid = model.predict_proba(X_valid)[:, 1]

pr_auc = average_precision_score(y_valid, proba_valid)
roc_auc = roc_auc_score(y_valid, proba_valid)
no_skill = y_valid.mean()

hasil = pd.DataFrame(
    {
        "metrik": [
            "PR-AUC (average precision)",
            "ROC-AUC",
            "PR-AUC no-skill (prevalensi validation)",
        ],
        "nilai": [pr_auc, roc_auc, no_skill],
    }
)
hasil["nilai"] = hasil["nilai"].round(4)
hasil

,metrik,nilai
0,PR-AUC (average precision),0.6458
1,ROC-AUC,0.8310
2,PR-AUC no-skill (prevalensi validation),0.2188


## Simpan artefak baseline

`model.joblib` berisi pipeline lengkap: imputer, scaler, one-hot encoder, dan
model regresi logistik. `metadata.json` mencatat versi, fitur masukan,
threshold sementara, metrik validation, tanggal latih, dan versi library.

In [5]:
import json

import joblib
import numpy as np
import sklearn

ARTIFACT_DIR = ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_VERSION = "baseline-logreg-v1"

metadata = {
    "model_version": MODEL_VERSION,
    "tipe_model": "LogisticRegression dengan fitur asli (baseline)",
    "tanggal_latih": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "target": "terlambat",
    "fitur_masukan": NUM_COLS + CAT_COLS,
    "fitur_numerik": NUM_COLS,
    "fitur_kategorikal": {
        kolom: sorted(str(nilai) for nilai in train[kolom].dropna().unique())
        for kolom in CAT_COLS
    },
    "split": {
        "train": {"sampai": "2025-12-31", "baris": len(train), "rasio_terlambat": float(y_train.mean())},
        "validation": {"mulai": "2026-01-01", "sampai": "2026-03-31", "baris": len(valid), "rasio_terlambat": float(y_valid.mean())},
        "test": {"mulai": "2026-04-01", "sampai": "2026-06-30", "baris": len(test), "rasio_terlambat": float(test["terlambat"].mean())},
    },
    "thresholds": {
        "kabari_pelanggan": 0.09,
        "prioritaskan": 0.30,
    },
    "threshold_status": "placeholder; 0,09 titik impas biaya, batas prioritaskan belum dikalibrasi",
    "metrics": {
        "validation": {
            "pr_auc": float(pr_auc),
            "roc_auc": float(roc_auc),
            "pr_auc_no_skill": float(no_skill),
        },
        "test": None,
    },
    "library_versions": {
        "python": sys.version.split()[0],
        "pandas": pd.__version__,
        "numpy": np.__version__,
        "scikit-learn": sklearn.__version__,
        "joblib": joblib.__version__,
    },
}

model_path = ARTIFACT_DIR / "model.joblib"
metadata_path = ARTIFACT_DIR / "metadata.json"

joblib.dump(model, model_path)
metadata_path.write_text(
    json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8"
)

print("Model   :", model_path)
print("Metadata:", metadata_path)
print("Versi   :", MODEL_VERSION)

Model   : C:\PROJECTVibeCoding\ModelMLStudyCase8\artifacts\model.joblib
Metadata: C:\PROJECTVibeCoding\ModelMLStudyCase8\artifacts\metadata.json
Versi   : baseline-logreg-v1


## Kesimpulan baseline

Baseline memakai fitur asli tanpa rekayasa fitur. Hasil validation di atas
menjadi acuan untuk dua model berikutnya. Test belum dievaluasi dan tidak
dipakai untuk memilih apa pun.